# Results Section — Numerical Verification Notebook v2

Verifies every numerical claim in the Results section by recomputing statistics **directly from source data**.

### Directory structure expected
```
project/
├── notebooks/          ← run this notebook from here
├── resources/
│   ├── models/simulated/   ← .pkl files (GLM.pkl, XGBoost.pkl, NN-*.pkl)
│   └── results/real/combined/
│       ├── combined_batches_0_1_2_3_all_r2.csv
│       ├── combined_batches_0_1_2_3_summary.csv
│       ├── combined_batches_0_1_2_3_wilcoxon_glm.csv
│       └── combined_batches_0_1_2_3_wilcoxon_mlp.csv
```

All statistics are recomputed from raw per-cell pR² values. Pre-computed CSVs are used as a secondary cross-check.

In [1]:
import os, sys

# Add project root so src imports work
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
from src.get_data import load_data

# Extract core arrays (squeeze to 1D)
X, Y, cell_ids, rec_ids = load_data("../resources/data/real/Temi_Data.mat")


print(f"X shape:        {X.shape}")
print(f"Y shape:        {Y.shape}")
print(f"cell_ids shape: {cell_ids.shape}")
print(f"rec_ids shape:  {rec_ids.shape}")
print(f"Total bins:     {len(Y):,}")

# ── Session-level verification ─────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"SESSION STRUCTURE VERIFICATION")
print(f"{'='*75}")
print(
    f"{'Session':>8}  {'Cells':>6}  {'Cell range':>14}  {'Bins/cell':>10}  "
    f"{'Total bins':>12}  {'Trials/cell':>12}"
)
print(f"{'-'*75}")

# Expected values from the table
expected = {
    1: {"n_cells": 70, "cell_range": (1, 70), "bins": 6230, "trials": 89},
    2: {"n_cells": 68, "cell_range": (71, 138), "bins": 6930, "trials": 99},
    3: {"n_cells": 44, "cell_range": (139, 182), "bins": 6790, "trials": 97},
    4: {"n_cells": 37, "cell_range": (183, 219), "bins": 7000, "trials": 100},
    5: {"n_cells": 28, "cell_range": (220, 247), "bins": 6860, "trials": 98},
}

BINS_PER_TRIAL = 70  # 3,500 ms / 50 ms bins
all_pass = True

for session_id in sorted(np.unique(rec_ids)):
    session_mask = rec_ids == session_id
    session_cells = np.unique(cell_ids[session_mask])

    n_cells = len(session_cells)
    cell_min = int(session_cells.min())
    cell_max = int(session_cells.max())

    # Bins per cell (should be same for all cells in session)
    bins_per_cell_vals = [np.sum((cell_ids == c) & session_mask) for c in session_cells]
    bins_per_cell = int(np.median(bins_per_cell_vals))
    total_bins = int(np.sum(session_mask))
    trials_per_cell = round(bins_per_cell / BINS_PER_TRIAL)

    print(
        f"{int(session_id):>8}  {n_cells:>6}  {cell_min:>6}–{cell_max:<6}  "
        f"{bins_per_cell:>10,}  {total_bins:>12,}  {trials_per_cell:>12}"
    )

    # Verify against expected
    exp = expected[int(session_id)]
    checks = [
        (
            n_cells == exp["n_cells"],
            f"Session {int(session_id)}: n_cells {n_cells} vs expected {exp['n_cells']}",
        ),
        (
            cell_min == exp["cell_range"][0],
            f"Session {int(session_id)}: cell_min {cell_min} vs expected {exp['cell_range'][0]}",
        ),
        (
            cell_max == exp["cell_range"][1],
            f"Session {int(session_id)}: cell_max {cell_max} vs expected {exp['cell_range'][1]}",
        ),
        (
            bins_per_cell == exp["bins"],
            f"Session {int(session_id)}: bins/cell {bins_per_cell} vs expected {exp['bins']}",
        ),
        (
            trials_per_cell == exp["trials"],
            f"Session {int(session_id)}: trials {trials_per_cell} vs expected {exp['trials']}",
        ),
    ]
    for ok, msg in checks:
        if not ok:
            print(f"  ❌ MISMATCH: {msg}")
            all_pass = False

print(f"{'-'*75}")

# ── Overall totals ─────────────────────────────────────────────────────────
total_cells = len(np.unique(cell_ids))
total_bins_check = len(Y)
expected_total = sum(e["n_cells"] * e["bins"] for e in expected.values())

print(f"\nTotal cells:     {total_cells}  (expected 247)")
print(f"Total bins:      {total_bins_check:,}  (expected {expected_total:,})")

checks_overall = [
    (total_cells == 247, f"Total cells: {total_cells} vs 247"),
    (
        total_bins_check == expected_total,
        f"Total bins: {total_bins_check:,} vs {expected_total:,}",
    ),
]
for ok, msg in checks_overall:
    if not ok:
        print(f"  ❌ MISMATCH: {msg}")
        all_pass = False

print(f"\n{'='*75}")
if all_pass:
    print("✅  ALL SESSION STRUCTURE CLAIMS VERIFIED CORRECT")
else:
    print("❌  SOME CLAIMS DO NOT MATCH — see above")
print(f"{'='*75}")

X shape:        (14, 1657180)
Y shape:        (1657180,)
cell_ids shape: (1657180,)
rec_ids shape:  (1657180,)
Total bins:     1,657,180

SESSION STRUCTURE VERIFICATION
 Session   Cells      Cell range   Bins/cell    Total bins   Trials/cell
---------------------------------------------------------------------------
       1      70       1–70           6,230       436,100            89
       2      68      71–138          6,930       471,240            99
       3      44     139–182          6,790       298,760            97
       4      37     183–219          7,000       259,000           100
       5      28     220–247          6,860       192,080            98
---------------------------------------------------------------------------

Total cells:     247  (expected 247)
Total bins:      1,657,180  (expected 1,657,180)

✅  ALL SESSION STRUCTURE CLAIMS VERIFIED CORRECT


In [15]:
import pickle
import numpy as np
from pathlib import Path

models_to_check = [
    "NN-PerCell-CNN",
    "NN-PerCell-RNN",
    "NN-PerCell-MLP",  # control — should not overfit
    "GLM",  # control — linear, cannot overfit in same way
]

all_results = {m: {"train": [], "test": []} for m in models_to_check}

# ── Collect results across all four batches ────────────────────────────────
for BATCH in range(4):
    MODEL_DIR = Path(f"../resources/models/real/batch_{BATCH}")

    for model_name in models_to_check:
        pkl_path = MODEL_DIR / f"{model_name}.pkl"
        if not pkl_path.exists():
            continue

        with open(pkl_path, "rb") as f:
            raw = pickle.load(f)

        if isinstance(raw, dict) and "results" in raw:
            results = raw["results"]
        else:
            results = raw
        results = {int(k): v for k, v in results.items()}

        for cell_id, cell_data in results.items():
            if isinstance(cell_data, dict):
                if "train" in cell_data and "pseudo_r2" in cell_data["train"]:
                    all_results[model_name]["train"].append(
                        cell_data["train"]["pseudo_r2"]
                    )
                if "test" in cell_data and "pseudo_r2" in cell_data["test"]:
                    all_results[model_name]["test"].append(
                        cell_data["test"]["pseudo_r2"]
                    )

# ── Per-batch breakdown ────────────────────────────────────────────────────
for BATCH in range(4):
    MODEL_DIR = Path(f"../resources/models/real/batch_{BATCH}")
    print(f"\n{'='*65}")
    print(f"OVERFITTING CHECK — Per-cell models, Batch {BATCH}")
    print(f"{'='*65}")
    print(
        f"\n{'Model':<30} {'Train pR²':>12} {'Test pR²':>12} {'Gap':>10} {'Verdict':>12}"
    )
    print(f"{'-'*65}")

    for model_name in models_to_check:
        pkl_path = MODEL_DIR / f"{model_name}.pkl"
        if not pkl_path.exists():
            print(f"  {model_name}: not found")
            continue

        with open(pkl_path, "rb") as f:
            raw = pickle.load(f)
        if isinstance(raw, dict) and "results" in raw:
            results = raw["results"]
        else:
            results = raw
        results = {int(k): v for k, v in results.items()}

        train_r2s, test_r2s = [], []
        for cell_data in results.values():
            if isinstance(cell_data, dict):
                if "train" in cell_data and "pseudo_r2" in cell_data["train"]:
                    train_r2s.append(cell_data["train"]["pseudo_r2"])
                if "test" in cell_data and "pseudo_r2" in cell_data["test"]:
                    test_r2s.append(cell_data["test"]["pseudo_r2"])

        if train_r2s and test_r2s:
            med_train = np.median(train_r2s)
            med_test = np.median(test_r2s)
            gap = med_train - med_test
            verdict = "OVERFIT" if (gap > 0.05 and med_test < 0.01) else "OK"
            print(
                f"  {model_name:<28} {med_train:>12.3f} {med_test:>12.3f} {gap:>10.3f} {verdict:>12}"
            )
        else:
            first = list(results.values())[0]
            print(
                f"  {model_name:<28}  keys not found — available: {list(first.keys()) if isinstance(first, dict) else type(first)}"
            )

# ── Pooled summary across all 100 cells ───────────────────────────────────
print(f"\n{'='*65}")
print(f"POOLED SUMMARY — All 4 batches (100 cells)")
print(f"{'='*65}")
print(f"\n{'Model':<30} {'Train pR²':>12} {'Test pR²':>12} {'Gap':>10} {'Verdict':>12}")
print(f"{'-'*65}")

for model_name in models_to_check:
    trains = all_results[model_name]["train"]
    tests = all_results[model_name]["test"]
    if trains and tests:
        med_train = np.median(trains)
        med_test = np.median(tests)
        gap = med_train - med_test
        verdict = "OVERFIT" if (gap > 0.05 and med_test < 0.01) else "OK"
        print(
            f"  {model_name:<28} {med_train:>12.3f} {med_test:>12.3f} {gap:>10.3f} {verdict:>12}"
        )

print(f"\n  Interpretation:")
print(f"  - Large gap + low test pR²  → OVERFITTING")
print(f"  - Both train and test low   → LEARNING FAILURE (inductive bias)")
print(f"  - MLP and GLM are controls  → small gap expected")


OVERFITTING CHECK — Per-cell models, Batch 0

Model                             Train pR²     Test pR²        Gap      Verdict
-----------------------------------------------------------------
  NN-PerCell-CNN                      0.001       -0.003      0.004           OK
  NN-PerCell-RNN                      0.050       -0.002      0.052      OVERFIT
  NN-PerCell-MLP                      0.068        0.023      0.045           OK
  GLM                                 0.021        0.007      0.014           OK

OVERFITTING CHECK — Per-cell models, Batch 1

Model                             Train pR²     Test pR²        Gap      Verdict
-----------------------------------------------------------------
  NN-PerCell-CNN                      0.003       -0.000      0.003           OK
  NN-PerCell-RNN                      0.046        0.025      0.021           OK
  NN-PerCell-MLP                      0.048        0.026      0.022           OK
  GLM                                 0.022  

In [4]:
import warnings
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")


# ── Paths (same relative structure as your other notebooks) ───────────────
ROOT = Path("..").resolve()
SIM_MODEL_DIR = ROOT / "resources" / "models" / "simulated"
REAL_CSV_DIR = ROOT / "resources" / "results" / "real" / "combined"

# ── Model name constants ───────────────────────────────────────────────────
SIM_PERCELL = ["GLM", "XGBoost", "NN-PerCell-MLP", "NN-PerCell-CNN", "NN-PerCell-RNN"]
SIM_TL = [
    "NN-DeepSharedShallowHead-TL-MLP",
    "NN-DeepSharedShallowHead-TL-CNN",
    "NN-DeepSharedShallowHead-TL-RNN",
    "NN-DeepSharedDeepHead-TL-MLP",
    "NN-DeepSharedDeepHead-TL-CNN",
    "NN-DeepSharedDeepHead-TL-RNN",
    "NN-ShallowSharedDeepHead-TL-MLP",
    "NN-ShallowSharedDeepHead-TL-CNN",
    "NN-ShallowSharedDeepHead-TL-RNN",
]
TL_MODELS = SIM_TL

SHORT = {
    "GLM": "GLM",
    "XGBoost": "XGBoost",
    "NN-PerCell-MLP": "MLP",
    "NN-PerCell-CNN": "CNN",
    "NN-PerCell-RNN": "RNN",
    "NN-DeepSharedDeepHead-TL-CNN": "DSD-CNN",
    "NN-DeepSharedDeepHead-TL-MLP": "DSD-MLP",
    "NN-DeepSharedDeepHead-TL-RNN": "DSD-RNN",
    "NN-DeepSharedShallowHead-TL-CNN": "DSS-CNN",
    "NN-DeepSharedShallowHead-TL-MLP": "DSS-MLP",
    "NN-DeepSharedShallowHead-TL-RNN": "DSS-RNN",
    "NN-ShallowSharedDeepHead-TL-CNN": "SSD-CNN",
    "NN-ShallowSharedDeepHead-TL-MLP": "SSD-MLP",
    "NN-ShallowSharedDeepHead-TL-RNN": "SSD-RNN",
}
BATCHES = {
    0: list(range(1, 26)),
    1: list(range(26, 51)),
    2: list(range(51, 76)),
    3: list(range(76, 101)),
}

print(f"Project root:            {ROOT}")
print(f"Simulated pkl dir found: {SIM_MODEL_DIR.exists()}")
print(f"Real CSV dir found:      {REAL_CSV_DIR.exists()}")

Project root:            C:\Users\Temitope Shitta\BioInfo Projects\BIOL61230_poisson_neural_net
Simulated pkl dir found: True
Real CSV dir found:      True


In [5]:
# ── Verification helpers ───────────────────────────────────────────────────
PASS = []
FAIL = []


def check(label, reported, actual, tol=0.001):
    """Check a continuous claim within tolerance."""
    try:
        ok = abs(float(reported) - float(actual)) <= tol
    except Exception:
        ok = False
    _log(label, reported, actual, ok)
    return ok


def check_exact(label, reported, actual):
    """Check an exact equality claim."""
    ok = reported == actual
    _log(label, reported, actual, ok)
    return ok


def _log(label, reported, actual, ok):
    icon = "✅" if ok else "❌"
    note = "" if ok else f"  ← actual = {actual}"
    print(f"  {icon}  {label}: {reported}{note}")
    (PASS if ok else FAIL).append(label)


def section(title):
    print(f'\n{"="*65}')
    print(f"  {title}")
    print(f'{"="*65}')

## 1. Load Simulated Data from PKL Files

In [6]:
def load_sim_model(name):
    """Load a simulated model pkl — exact pattern from results_figures.ipynb."""
    pkl_path = SIM_MODEL_DIR / f"{name}.pkl"
    if not pkl_path.exists():
        print(f"  [SKIP] {name} — file not found at {pkl_path}")
        return None
    try:
        with open(pkl_path, "rb") as f:
            raw = pickle.load(f)

        # The pkl may be a list of (dict, name) tuples from run_experiment
        if isinstance(raw, list):
            raw = raw[0][0] if isinstance(raw[0], tuple) else raw[0]

        # Unwrap the 'results' key if present
        if isinstance(raw, dict) and "results" in raw:
            results = raw["results"]
        else:
            results = raw

        # Normalise cell IDs to int (simulated uses floats: 0.0, 1.0, ...)
        results = {int(k): v for k, v in results.items()}
        return results

    except Exception as e:
        print(f"  [WARN] {name}: {e}")
        return None


# Load all simulated models
print("Loading simulated pkl files...")
sim_r2 = {}  # {short_name: np.array of per-cell test pR²}

for name in SIM_PERCELL + SIM_TL:
    results = load_sim_model(name)
    short = SHORT[name]
    if results is not None:
        vals = [
            v["test"]["pseudo_r2"]
            for v in results.values()
            if isinstance(v, dict) and "test" in v
        ]
        sim_r2[short] = np.array(vals)
        print(f"  ✅  {short:<12}  n={len(vals)}  median={np.median(vals):.3f}")
    else:
        sim_r2[short] = None
        print(f"  ⚠️   {short:<12}  NOT FOUND — simulated claims will be skipped")

loaded = sum(v is not None for v in sim_r2.values())
print(f"\nLoaded {loaded}/{len(sim_r2)} simulated models")

Loading simulated pkl files...
  ✅  GLM           n=20  median=0.647
  ✅  XGBoost       n=20  median=0.710
  ✅  MLP           n=20  median=0.821
  ✅  CNN           n=20  median=0.841
  ✅  RNN           n=20  median=0.877
  ✅  DSS-MLP       n=20  median=0.610
  ✅  DSS-CNN       n=20  median=0.765
  ✅  DSS-RNN       n=20  median=0.663
  ✅  DSD-MLP       n=20  median=0.890
  ✅  DSD-CNN       n=20  median=0.882
  ✅  DSD-RNN       n=20  median=0.910
  ✅  SSD-MLP       n=20  median=0.878
  ✅  SSD-CNN       n=20  median=0.701
  ✅  SSD-RNN       n=20  median=0.858

Loaded 14/14 simulated models


## 2. Load Real Data from CSVs

In [7]:
# Load pre-computed CSVs
df_all = pd.read_csv(
    REAL_CSV_DIR / "combined_batches_0_1_2_3_all_r2.csv", index_col="cell"
)
df_sum = pd.read_csv(
    REAL_CSV_DIR / "combined_batches_0_1_2_3_summary.csv", index_col="model"
)
df_wglm = pd.read_csv(
    REAL_CSV_DIR / "combined_batches_0_1_2_3_wilcoxon_glm.csv", index_col="model"
)
df_wmlp = pd.read_csv(
    REAL_CSV_DIR / "combined_batches_0_1_2_3_wilcoxon_mlp.csv", index_col="model"
)

print(f"Real data: {len(df_all)} cells × {len(df_all.columns)} models")
print(f"Cell range: {df_all.index.min()} – {df_all.index.max()}")
print(f"\nModels available:")
for c in df_all.columns:
    print(f"  {c}")

Real data: 100 cells × 14 models
Cell range: 1 – 100

Models available:
  GLM
  NN-DeepSharedDeepHead-TL-CNN
  NN-DeepSharedDeepHead-TL-MLP
  NN-DeepSharedDeepHead-TL-RNN
  NN-DeepSharedShallowHead-TL-CNN
  NN-DeepSharedShallowHead-TL-MLP
  NN-DeepSharedShallowHead-TL-RNN
  NN-PerCell-CNN
  NN-PerCell-MLP
  NN-PerCell-RNN
  NN-ShallowSharedDeepHead-TL-CNN
  NN-ShallowSharedDeepHead-TL-MLP
  NN-ShallowSharedDeepHead-TL-RNN
  XGBoost


## 3. Recompute All Statistics from Raw pR² Values

In [8]:
def wilcoxon_one_sided(a, b):
    """One-sided Wilcoxon signed-rank test: H1: a > b."""
    diff = np.array(a) - np.array(b)
    if np.all(diff == 0):
        return np.nan, 1.0
    return wilcoxon(diff, alternative="greater")


# Recompute pooled (100-cell) Wilcoxon stats
print("Recomputing Wilcoxon statistics from raw pR² values...")
recomputed = {}
glm_vals = df_all["GLM"].values
mlp_vals = df_all["NN-PerCell-MLP"].values

for col in df_all.columns:
    if col == "GLM":
        continue
    model_vals = df_all[col].values
    W_glm, p_glm = wilcoxon_one_sided(model_vals, glm_vals)
    median = np.median(model_vals)
    pct_pos = (model_vals > 0).mean() * 100
    recomputed[col] = {
        "median": median,
        "pct": pct_pos,
        "W_glm": W_glm,
        "p_glm": p_glm,
    }
    if col in [m for m in TL_MODELS]:
        W_mlp, p_mlp = wilcoxon_one_sided(model_vals, mlp_vals)
        recomputed[col]["W_mlp"] = W_mlp
        recomputed[col]["p_mlp"] = p_mlp

# Add GLM
recomputed["GLM"] = {
    "median": np.median(glm_vals),
    "pct": (glm_vals > 0).mean() * 100,
}

print("Done. Sample check:")
print(f"  GLM median (recomputed):     {recomputed['GLM']['median']:.4f}")
print(f"  GLM median (CSV):            {df_sum.loc['GLM','median']:.4f}")
print(
    f"  DSD-CNN W vs GLM (recomp):   {recomputed['NN-DeepSharedDeepHead-TL-CNN']['W_glm']:.0f}"
)
print(
    f"  DSD-CNN W vs GLM (CSV):      {df_wglm.loc['NN-DeepSharedDeepHead-TL-CNN','W']:.0f}"
)

Recomputing Wilcoxon statistics from raw pR² values...
Done. Sample check:
  GLM median (recomputed):     0.0124
  GLM median (CSV):            0.0124
  DSD-CNN W vs GLM (recomp):   4167
  DSD-CNN W vs GLM (CSV):      4167


## 4. Section 3.1 — Simulated Data Claims

In [9]:
section("SECTION 3.1 — SIMULATED DATA (from pkl files)")

claims_31 = [
    # Per-cell models
    ("GLM median", "GLM", 0.647),
    ("XGBoost median", "XGBoost", 0.710),
    ("MLP median", "MLP", 0.821),
    ("CNN median", "CNN", 0.841),
    ("RNN median", "RNN", 0.877),
    # DSD family
    ("DSD-RNN median (best overall)", "DSD-RNN", 0.910),
    ("DSD-MLP median", "DSD-MLP", 0.890),
    ("DSD-CNN median", "DSD-CNN", 0.882),
    # SSD family
    ("SSD-MLP median", "SSD-MLP", 0.878),
    ("SSD-RNN median", "SSD-RNN", 0.858),
    ("SSD-CNN median", "SSD-CNN", 0.701),
    # DSS family — ShallowHead reversal
    ("DSS-MLP median (below per-cell MLP baseline)", "DSS-MLP", 0.610),
    ("DSS-RNN median (below per-cell MLP baseline)", "DSS-RNN", 0.663),
    ("DSS-CNN median", "DSS-CNN", 0.765),
]

print()
for label, short, reported in claims_31:
    arr = sim_r2.get(short)
    if arr is None:
        print(f"  ⚠️   {label}: pkl not found — cannot verify")
        continue
    actual = np.median(arr)
    check(label, reported, actual)

# Verify DSD-RNN is actually the highest median
all_sim_medians = {k: np.median(v) for k, v in sim_r2.items() if v is not None}
best = max(all_sim_medians, key=all_sim_medians.get)
print()
check_exact("DSD-RNN is highest median model", "DSD-RNN", best)

# Verify DSS models below per-cell MLP
if sim_r2.get("MLP") is not None:
    mlp_med = np.median(sim_r2["MLP"])
    for dss in ["DSS-MLP", "DSS-RNN", "DSS-CNN"]:
        if sim_r2.get(dss) is not None:
            below = np.median(sim_r2[dss]) < mlp_med
            check(f"{dss} below per-cell MLP ({mlp_med:.3f})", True, below)


  SECTION 3.1 — SIMULATED DATA (from pkl files)

  ✅  GLM median: 0.647
  ✅  XGBoost median: 0.71
  ✅  MLP median: 0.821
  ✅  CNN median: 0.841
  ✅  RNN median: 0.877
  ✅  DSD-RNN median (best overall): 0.91
  ✅  DSD-MLP median: 0.89
  ✅  DSD-CNN median: 0.882
  ✅  SSD-MLP median: 0.878
  ✅  SSD-RNN median: 0.858
  ✅  SSD-CNN median: 0.701
  ✅  DSS-MLP median (below per-cell MLP baseline): 0.61
  ✅  DSS-RNN median (below per-cell MLP baseline): 0.663
  ✅  DSS-CNN median: 0.765

  ✅  DSD-RNN is highest median model: DSD-RNN
  ✅  DSS-MLP below per-cell MLP (0.821): True
  ✅  DSS-RNN below per-cell MLP (0.821): True
  ✅  DSS-CNN below per-cell MLP (0.821): True


## 5. Section 3.2 — Baseline and Per-cell Models (Real Data)

In [10]:
section("SECTION 3.2 — REAL DATA: BASELINES AND PER-CELL")

print("\n--- GLM ---")
check("GLM median pR²", 0.012, recomputed["GLM"]["median"])
check_exact("GLM % above chance", 71, round(recomputed["GLM"]["pct"]))

print("\n--- XGBoost vs GLM ---")
xgb = recomputed["XGBoost"]
check("XGBoost median", 0.034, xgb["median"])
check_exact("XGBoost % above chance", 72, round(xgb["pct"]))
check_exact("XGBoost W vs GLM", 3649, int(xgb["W_glm"]))
check("XGBoost p vs GLM < 0.001", True, xgb["p_glm"] < 0.001)

print("\n--- Per-cell MLP vs GLM ---")
mlp = recomputed["NN-PerCell-MLP"]
check("MLP median", 0.021, mlp["median"])
check_exact("MLP % above chance", 72, round(mlp["pct"]))
check_exact("MLP W vs GLM", 3624, int(mlp["W_glm"]))
check("MLP p vs GLM < 0.001", True, mlp["p_glm"] < 0.001)

print("\n--- Per-cell CNN vs GLM (expect FAILURE / ns) ---")
cnn = recomputed["NN-PerCell-CNN"]
check("CNN median", -0.001, cnn["median"])
check_exact("CNN % above chance", 40, round(cnn["pct"]))
check_exact("CNN W vs GLM", 1793, int(cnn["W_glm"]))
check("CNN p vs GLM ≈ 0.994", 0.994, cnn["p_glm"])

print("\n--- Per-cell RNN vs GLM (expect FAILURE / ns) ---")
rnn = recomputed["NN-PerCell-RNN"]
check("RNN median", 0.005, rnn["median"])
check_exact("RNN % above chance", 53, round(rnn["pct"]))
check_exact("RNN W vs GLM", 2604, int(rnn["W_glm"]))
check("RNN p vs GLM ≈ 0.393", 0.393, rnn["p_glm"])


  SECTION 3.2 — REAL DATA: BASELINES AND PER-CELL

--- GLM ---
  ✅  GLM median pR²: 0.012
  ✅  GLM % above chance: 71

--- XGBoost vs GLM ---
  ✅  XGBoost median: 0.034
  ✅  XGBoost % above chance: 72
  ✅  XGBoost W vs GLM: 3649
  ✅  XGBoost p vs GLM < 0.001: True

--- Per-cell MLP vs GLM ---
  ✅  MLP median: 0.021
  ✅  MLP % above chance: 72
  ✅  MLP W vs GLM: 3624
  ✅  MLP p vs GLM < 0.001: True

--- Per-cell CNN vs GLM (expect FAILURE / ns) ---
  ✅  CNN median: -0.001
  ✅  CNN % above chance: 40
  ✅  CNN W vs GLM: 1793
  ✅  CNN p vs GLM ≈ 0.994: 0.994

--- Per-cell RNN vs GLM (expect FAILURE / ns) ---
  ✅  RNN median: 0.005
  ✅  RNN % above chance: 53
  ✅  RNN W vs GLM: 2604
  ✅  RNN p vs GLM ≈ 0.393: 0.393


True

## 6. Section 3.3 — Transfer Learning Results

In [11]:
section("SECTION 3.3 — TL ARCHITECTURES (REAL DATA)")

# All 9 TL models sig vs GLM
print("\n--- All 9 TL models significant vs GLM (p < 0.001) ---")
n_sig_glm = 0
for name in TL_MODELS:
    p = recomputed[name]["p_glm"]
    sig = p < 0.001
    if sig:
        n_sig_glm += 1
    print(
        f'  {"✅" if sig else "❌"}  {SHORT[name]:<10}  p={p:.2e}  {"***" if sig else "ns"}'
    )
check_exact("All 9 TL models sig vs GLM", 9, n_sig_glm)

# Ranked medians and % above chance
print("\n--- Ranked model performance ---")
ranked_claims = [
    ("DSD-CNN", 0.044, 79),
    ("DSS-CNN", 0.040, 79),
    ("SSD-RNN", 0.038, 76),
    ("DSD-MLP", 0.037, 80),
    ("DSD-RNN", 0.036, 75),
    ("SSD-MLP", 0.036, 78),
    ("SSD-CNN", 0.032, 76),
    ("DSS-MLP", 0.030, 78),
    ("DSS-RNN", 0.029, 70),
]
# Find full name from short
short_to_full = {v: k for k, v in SHORT.items()}
for short, med, pct in ranked_claims:
    full = short_to_full[short]
    check(f"{short} median", med, recomputed[full]["median"])
    check_exact(f"{short} % above chance", pct, round(recomputed[full]["pct"]))

# 8/9 TL sig vs MLP
print("\n--- 8 of 9 TL models significant vs per-cell MLP ---")
n_sig_mlp = 0
for name in TL_MODELS:
    p = recomputed[name].get("p_mlp", np.nan)
    sig = p < 0.05
    if sig:
        n_sig_mlp += 1
    status = "✅" if sig else "  "
    print(f'  {status}  {SHORT[name]:<10}  p={p:.4f}  {"sig" if sig else "ns"}')
check_exact("8 of 9 TL significant vs MLP", 8, n_sig_mlp)

# DSS-RNN exception
print("\n--- DSS-RNN exception (ns vs MLP) ---")
dss_rnn = "NN-DeepSharedShallowHead-TL-RNN"
check_exact("DSS-RNN W vs MLP", 2891, int(recomputed[dss_rnn]["W_mlp"]))
check("DSS-RNN p vs MLP ≈ 0.104", 0.104, recomputed[dss_rnn]["p_mlp"])

# 76% cells best TL > GLM
print("\n--- 76% of cells: best TL > GLM ---")
best_tl_col = "NN-DeepSharedDeepHead-TL-CNN"
pct_better = (df_all[best_tl_col] > df_all["GLM"]).mean() * 100
check("76% of cells TL-DSD-CNN > GLM", 76, pct_better)


  SECTION 3.3 — TL ARCHITECTURES (REAL DATA)

--- All 9 TL models significant vs GLM (p < 0.001) ---
  ✅  DSS-MLP     p=6.88e-10  ***
  ✅  DSS-CNN     p=1.37e-10  ***
  ✅  DSS-RNN     p=5.11e-05  ***
  ✅  DSD-MLP     p=3.71e-12  ***
  ✅  DSD-CNN     p=8.22e-09  ***
  ✅  DSD-RNN     p=4.64e-07  ***
  ✅  SSD-MLP     p=2.53e-12  ***
  ✅  SSD-CNN     p=3.91e-06  ***
  ✅  SSD-RNN     p=1.94e-07  ***
  ✅  All 9 TL models sig vs GLM: 9

--- Ranked model performance ---
  ✅  DSD-CNN median: 0.044
  ✅  DSD-CNN % above chance: 79
  ✅  DSS-CNN median: 0.04
  ✅  DSS-CNN % above chance: 79
  ✅  SSD-RNN median: 0.038
  ✅  SSD-RNN % above chance: 76
  ✅  DSD-MLP median: 0.037
  ✅  DSD-MLP % above chance: 80
  ✅  DSD-RNN median: 0.036
  ✅  DSD-RNN % above chance: 75
  ✅  SSD-MLP median: 0.036
  ✅  SSD-MLP % above chance: 78
  ✅  SSD-CNN median: 0.032
  ✅  SSD-CNN % above chance: 76
  ✅  DSS-MLP median: 0.03
  ✅  DSS-MLP % above chance: 78
  ✅  DSS-RNN median: 0.029
  ✅  DSS-RNN % above chance: 70

--

True

## 7. Section 3.4 — Batch-Level Analysis

In [12]:
section("SECTION 3.4 — BATCH-LEVEL ANALYSIS (recomputed from per-cell pR²)")

print("\n--- Per-batch GLM medians ---")
batch_glm_expected = {0: 0.007, 1: 0.018, 2: 0.003, 3: 0.021}
for b, exp in batch_glm_expected.items():
    actual = df_all.loc[BATCHES[b], "GLM"].median()
    check(f"Batch {b} GLM median (session {1 if b<=1 else 2})", exp, actual)

print("\n--- Batch 1: all 9 TL models p < 0.001 vs GLM ---")
b1_glm = df_all.loc[BATCHES[1], "GLM"].values
n_b1_glm = sum(
    1
    for m in TL_MODELS
    if wilcoxon_one_sided(df_all.loc[BATCHES[1], m].values, b1_glm)[1] < 0.001
)
check_exact("Batch 1: all 9 TL sig vs GLM (p<0.001)", 9, n_b1_glm)

print("\n--- Batch 1: 8 of 9 TL models p < 0.001 vs MLP ---")
b1_mlp = df_all.loc[BATCHES[1], "NN-PerCell-MLP"].values
n_b1_mlp = sum(
    1
    for m in TL_MODELS
    if wilcoxon_one_sided(df_all.loc[BATCHES[1], m].values, b1_mlp)[1] < 0.001
)
check_exact("Batch 1: 8 of 9 TL sig vs MLP (p<0.001)", 8, n_b1_mlp)

print("\n--- Batch 2 (cross-session): XGBoost and MLP ns vs GLM ---")
b2_glm = df_all.loc[BATCHES[2], "GLM"].values
for short_name, full_name in [
    ("XGBoost", "XGBoost"),
    ("per-cell MLP", "NN-PerCell-MLP"),
]:
    _, p = wilcoxon_one_sided(df_all.loc[BATCHES[2], full_name].values, b2_glm)
    check(f"Batch 2: {short_name} ns vs GLM (p={p:.3f})", True, p >= 0.05)

print("\n--- Batch 2: TL models significant vs GLM ---")
b2_sig = []
for m in TL_MODELS:
    _, p = wilcoxon_one_sided(df_all.loc[BATCHES[2], m].values, b2_glm)
    sig = p < 0.05
    if sig:
        b2_sig.append(SHORT[m])
    print(f'  {"✅" if sig else "  "}  {SHORT[m]:<10}  p={p:.4f}')
print(f'\n  Significant: {len(b2_sig)}/9 ({", ".join(b2_sig)})')

# ⚠️ KEY CHECK: report says 5/9 but data may show different
print()
if len(b2_sig) == 7:
    print(
        '  ⚠️  DISCREPANCY: Report text says "5 of 9" but data shows 7 of 9 significant.'
    )
    print(
        '     CORRECTION NEEDED: change "five of nine" to "seven of nine" in Section 3.4'
    )
    FAIL.append(
        "3.4 Batch 2 count: report says 5/9 but data shows 7/9 — NEEDS CORRECTION"
    )
elif len(b2_sig) == 5:
    check_exact("Batch 2: 5 of 9 TL sig vs GLM", 5, len(b2_sig))
else:
    print(
        f"  ℹ️  Batch 2 significant count: {len(b2_sig)}/9 — verify against report text"
    )

print("\n--- Batch 3 (session 2): no TL model sig vs per-cell MLP ---")
b3_mlp = df_all.loc[BATCHES[3], "NN-PerCell-MLP"].values
b3_ns = sum(
    1
    for m in TL_MODELS
    if wilcoxon_one_sided(df_all.loc[BATCHES[3], m].values, b3_mlp)[1] >= 0.05
)
check_exact("Batch 3: all 9 TL ns vs MLP", 9, b3_ns)


  SECTION 3.4 — BATCH-LEVEL ANALYSIS (recomputed from per-cell pR²)

--- Per-batch GLM medians ---
  ✅  Batch 0 GLM median (session 1): 0.007
  ✅  Batch 1 GLM median (session 1): 0.018
  ✅  Batch 2 GLM median (session 2): 0.003
  ✅  Batch 3 GLM median (session 2): 0.021

--- Batch 1: all 9 TL models p < 0.001 vs GLM ---
  ✅  Batch 1: all 9 TL sig vs GLM (p<0.001): 9

--- Batch 1: 8 of 9 TL models p < 0.001 vs MLP ---
  ❌  Batch 1: 8 of 9 TL sig vs MLP (p<0.001): 8  ← actual = 9

--- Batch 2 (cross-session): XGBoost and MLP ns vs GLM ---
  ✅  Batch 2: XGBoost ns vs GLM (p=0.521): True
  ✅  Batch 2: per-cell MLP ns vs GLM (p=0.625): True

--- Batch 2: TL models significant vs GLM ---
  ✅  DSS-MLP     p=0.0101
  ✅  DSS-CNN     p=0.0211
      DSS-RNN     p=0.4060
  ✅  DSD-MLP     p=0.0015
  ✅  DSD-CNN     p=0.0294
  ✅  DSD-RNN     p=0.0401
  ✅  SSD-MLP     p=0.0013
      SSD-CNN     p=0.1050
  ✅  SSD-RNN     p=0.0159

  Significant: 7/9 (DSS-MLP, DSS-CNN, DSD-MLP, DSD-CNN, DSD-RNN, SSD-ML

True

## 8. Cross-Check: Recomputed vs Pre-computed CSVs

In [13]:
section("CROSS-CHECK: Recomputed statistics vs pre-computed CSV values")

print("\nChecking recomputed medians match summary CSV...")
mismatches = []
for col in df_all.columns:
    if col in df_sum.index:
        recomp = np.median(df_all[col].values)
        csv_val = df_sum.loc[col, "median"]
        if abs(recomp - csv_val) > 1e-8:
            mismatches.append((col, recomp, csv_val))

if mismatches:
    print(f"  ❌ {len(mismatches)} median mismatches between recomputed and CSV:")
    for name, r, c in mismatches:
        print(f"     {name}: recomputed={r:.6f}, CSV={c:.6f}")
else:
    print(f"  ✅ All {len(df_all.columns)} medians match CSV exactly")

print("\nChecking recomputed W-statistics match Wilcoxon CSV...")
w_mismatches = []
for col in df_all.columns:
    if col == "GLM":
        continue
    if col in df_wglm.index:
        W_recomp = recomputed.get(col, {}).get("W_glm", np.nan)
        W_csv = df_wglm.loc[col, "W"]
        if not np.isnan(W_recomp) and abs(W_recomp - W_csv) > 0.5:
            w_mismatches.append((col, W_recomp, W_csv))

if w_mismatches:
    print(f"  ❌ {len(w_mismatches)} W-statistic mismatches:")
    for name, r, c in w_mismatches:
        print(f"     {name}: recomputed={r:.0f}, CSV={c:.0f}")
else:
    print(f"  ✅ All W-statistics match CSV exactly")


  CROSS-CHECK: Recomputed statistics vs pre-computed CSV values

Checking recomputed medians match summary CSV...
  ✅ All 14 medians match CSV exactly

Checking recomputed W-statistics match Wilcoxon CSV...
  ✅ All W-statistics match CSV exactly


## 9. Full Results Summary

In [14]:
section("FULL VERIFICATION SUMMARY")

total = len(PASS) + len(FAIL)
print(f"\n  Total claims checked: {total}")
print(f"  ✅ PASS: {len(PASS)}")
print(f"  ❌ FAIL: {len(FAIL)}")

if FAIL:
    print("\n  Failed claims:")
    for f in FAIL:
        print(f"    ✗ {f}")
else:
    print("\n  ✅ ALL CLAIMS VERIFIED CORRECT")

print("\n  Note: simulated data claims require pkl files in")
print(f"  {SIM_MODEL_DIR}")


  FULL VERIFICATION SUMMARY

  Total claims checked: 69
  ✅ PASS: 67
  ❌ FAIL: 2

  Failed claims:
    ✗ Batch 1: 8 of 9 TL sig vs MLP (p<0.001)
    ✗ 3.4 Batch 2 count: report says 5/9 but data shows 7/9 — NEEDS CORRECTION

  Note: simulated data claims require pkl files in
  C:\Users\Temitope Shitta\BioInfo Projects\BIOL61230_poisson_neural_net\resources\models\simulated
